# 🚗 Bilvärderaren — pipeline

En enda notebook, cell för cell: **läs dataset → städa → modellera → utvärdera → output.**

- **Källa:** `dataset/car_price_dataset.csv` (10 000 bilar). Ingen scraping, ingen databas.
- **Mål (`price_sek`):** annonspriset. Datasetets `Price` saknar valuta — vi antar EUR och räknar
  om till SEK med en fast kurs (`EUR_SEK`, en kalibreringsratt du kan justera).
- **Vad vi skickar ut:** värdering (privat/handlare), fyndanalys, modelljämförelse och feature importance.

## 0. Beroenden & konstanter

Alla justerbara antaganden samlade på ett ställe.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV = "dataset/car_price_dataset.csv"

# Datasetets Price saknar valuta -> anta EUR, räkna om till SEK. Justera vid behov.
EUR_SEK = 11.5
# Handlare tar påslag (garanti/marginal) -> privatköp är billigare. Datasetet saknar
# säljartyp, så gapet sätts deterministiskt här istället för att läras från data.
PRIVATE_MARGIN = 0.13
# Fynd-tröskel: >=10 % under marknadsvärde = fynd, >=10 % över = dyr.
DEAL_THRESHOLD = 0.10

## 1. Läs in datasetet

In [ ]:
raw = pd.read_csv(CSV, sep=";")
print(f"{len(raw)} rader, {raw.shape[1]} kolumner")
raw.head()

## 2. Städa & utforska

Kolla dubletter och saknade värden, ta bort dubletter och räkna om priset till SEK.

In [ ]:
print("Dubletter:", raw.duplicated().sum())
print("Saknade värden per kolumn:")
print(raw.isna().sum())

df = raw.drop_duplicates().reset_index(drop=True)
df["price_sek"] = (df["Price"] * EUR_SEK).round().astype(int)
df.describe(include="all").T

## 3. Features & target

Kategoriska features one-hot-kodas, numeriska används som de är. `price_sek` är target.

In [ ]:
CAT = ["Brand", "Model", "Fuel_Type", "Transmission"]
NUM = ["Year", "Engine_Size", "Mileage", "Doors", "Owner_Count"]
TARGET = "price_sek"

X, y = df[CAT + NUM], df[TARGET]

def make_pipe(model):
    """Preprocessing (one-hot på kategoriska, passthrough på numeriska) + modell."""
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("num", "passthrough", NUM),
    ])
    return Pipeline([("prep", prep), ("model", model)])

## 4. Träna & utvärdera (80/20-split)

RandomForest på en train/test-split. MAE = snittfel i kr, R² = andel förklarad varians.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = make_pipe(RandomForestRegressor(n_estimators=200, random_state=42))
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print(f"RandomForest — MAE={mean_absolute_error(y_test, pred):,.0f} kr, "
      f"R2={r2_score(y_test, pred):.3f}")

## 5. Modelljämförelse: RandomForest vs LinearRegression

Samma preprocessing och indata för båda. Per modell: MAE/MSE/RMSE på test-splitten plus
5-fold korsvalidering (MAE-medel ± std) på hela datan — mer tillförlitligt än en enda split.
Vinnaren (lägst CV-MAE) refit:as på **all** data och blir modellen vi predikterar med.

In [ ]:
def new_model(name):
    return (RandomForestRegressor(n_estimators=200, random_state=42)
            if name == "RandomForest" else LinearRegression())

rows = {}
for name in ("RandomForest", "LinearRegression"):
    pipe = make_pipe(new_model(name))
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    mse = mean_squared_error(y_test, p)
    cv = -cross_val_score(make_pipe(new_model(name)), X, y, cv=5,
                          scoring="neg_mean_absolute_error")
    rows[name] = {
        "MAE": mean_absolute_error(y_test, p), "MSE": mse, "RMSE": mse ** 0.5,
        "CV-MAE": cv.mean(), "CV-std": cv.std(),
    }

comparison = pd.DataFrame(rows).T
winner = comparison["CV-MAE"].idxmin()
print(f"Vinnare (lägst CV-MAE): {winner}")

# Refit vinnaren på ALL data -> modellen vi värderar med nedan.
model = make_pipe(new_model(winner)).fit(X, y)
comparison.round(0)

## 6. Feature importance

Vilka features driver priset? Vi läser av en RandomForest (trädmodeller ger importance
direkt) och summerar one-hot-kolumnerna tillbaka till ursprungskolumnen.

In [ ]:
imp_rf = make_pipe(RandomForestRegressor(n_estimators=200, random_state=42)).fit(X, y)
names = imp_rf.named_steps["prep"].get_feature_names_out()
importances = pd.Series(imp_rf.named_steps["model"].feature_importances_, index=names)

def to_original(feature_name):
    body = feature_name.split("__", 1)[1]  # 'cat__Brand_Kia' -> 'Brand_Kia'
    for col in CAT + NUM:
        if body == col or body.startswith(col + "_"):
            return col
    return body

by_feature = importances.groupby(to_original).sum().sort_values(ascending=False)
by_feature.round(3)

## 7. Output — värdering (privat vs handlare)

Modellens prediktion = marknadsvärde (handlarnivå). Privatpris = `PRIVATE_MARGIN` under,
eftersom handlare tar påslag. Datasetet saknar säljartyp, så gapet är ett antagande.

In [ ]:
def estimate(car: dict) -> dict:
    """{'privat': kr, 'handlare': kr} för en bil. `car` måste ha alla CAT+NUM-fält."""
    base = model.predict(pd.DataFrame([car])[CAT + NUM])[0]
    return {"privat": int(round(base * (1 - PRIVATE_MARGIN))),
            "handlare": int(round(base))}

example = {"Brand": "Toyota", "Model": "Corolla", "Year": 2019, "Engine_Size": 2.0,
           "Fuel_Type": "Petrol", "Transmission": "Automatic", "Mileage": 60000,
           "Doors": 4, "Owner_Count": 1}
estimate(example)

## 8. Output — fyndanalys

Tre vinklar för en fyndjägare: är *ett* pris ett fynd (`deal`), vad kostar liknande bilar
(`comparables`), och vilka annonser i datan är mest underprissatta (`rank_deals`).

*Obs:* `rank_deals` predikterar in-sample (raderna finns i träningsdatan), så residualerna
är optimistiskt små — duger för relativa fynd i ett skolprojekt.

In [ ]:
def deal(car: dict, asking: int) -> dict:
    predicted = model.predict(pd.DataFrame([car])[CAT + NUM])[0]
    pct = (predicted - asking) / predicted
    verdict = ("fynd" if pct >= DEAL_THRESHOLD
               else "dyr" if pct <= -DEAL_THRESHOLD else "marknadspris")
    return {"predicted": int(predicted), "asking": asking,
            "pct_below_market": round(pct, 3), "verdict": verdict}

def comparables(brand: str, car_model: str) -> dict:
    same = df[(df["Brand"] == brand) & (df["Model"] == car_model)]["price_sek"]
    if same.empty:
        return {"n": 0}
    return {"n": int(same.size), "median": int(same.median()),
            "min": int(same.min()), "max": int(same.max())}

print("deal:", deal(example, 150_000))
print("comparables:", comparables("Toyota", "Corolla"))

In [ ]:
def rank_deals(top: int = 10) -> pd.DataFrame:
    out = df.copy()
    out["market_price"] = model.predict(X)
    out["pct_below_market"] = ((out["market_price"] - out["price_sek"])
                                / out["market_price"]).round(3)
    cols = ["Brand", "Model", "Year", "Mileage", "price_sek",
            "market_price", "pct_below_market"]
    return out.sort_values("pct_below_market", ascending=False)[cols].head(top)

rank_deals()